# Частица из A в B: диффузия траекторий

Этот ноутбук использует модули проекта, без повторения реализации моделей и семплеров. Меняйте `DIM = 2` на `3`: меняются форма траектории, условия, стены и визуализации.

Короткий запуск на CPU проверяет рабочий цикл. Три эпохи — проверка интерфейсов; для качественного обхода препятствий увеличьте выборку и число эпох. Эталонные маршруты строятся A*, а качество сгенерированных маршрутов проверяется отдельно.


In [ ]:
from pathlib import Path
import sys

# Одинаково работает при cwd в корне проекта или в notebooks/.
ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "diffusion_lab").is_dir()
)
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch
from IPython.display import Image, display
from examples.navigation_research import run_navigation, render_navigation
from diffusion_lab.environments import WallNavigation
from diffusion_lab.visualization import plot_trajectories

torch.set_num_threads(min(torch.get_num_threads(), 4))
DIM = 2
SEED = 7
OUTPUT = ROOT / "outputs" / f"notebook-navigation-{DIM}d"


## Условия и обучение

Пример обучает небольшой Transformer, который видит позиции внутри одной траектории. Conditioner `mlp` получает непрерывный вектор: `start(D), goal(D), walls(W, 2, D)`. В нём координаты A/B и нижний/верхний углы каждого препятствия; размер вычисляется из датасета.

К регрессии шума добавлены штрафы `endpoint`, `collision`, `smoothness`, `bounds`. Настройки находятся в `examples/navigation_research.py`; новые лоссы и методы регистрируются через модули `losses` и `objectives`.


In [ ]:
experiment = run_navigation(
    dim=DIM, n_samples=64, epochs=3, seed=SEED,
    device="cpu", path_length=32, batch_size=16,
)
print("Sample shape:", experiment.dataset.sample_shape)
print("Continuous condition dimension:", experiment.dataset.condition_dim)
print("Condition layout:", experiment.dataset.metadata["condition_layout"])
assert experiment.metrics["reference_collision_free_fraction"] == 1.0
assert experiment.metrics["maximum_endpoint_error"] == 0.0


## A/B и проверка стен

При семплировании известны только первый и последний waypoints. Их маска передаётся как `known_mask`, значения — как `known_values`; pipeline подставляет соответствующий уровень шума на каждом шаге и точные A/B на последнем.

Фиксация концов не гарантирует обхода стен. Метрика `generated_collision_free_fraction` проверяет все отрезки между waypoints, расширенные стены и границы области. Тонкую стенку нельзя перескочить только потому, что соседние waypoints лежат снаружи.


In [ ]:
conditions = experiment.conditions
known_values = torch.zeros_like(experiment.dataset.samples[:len(conditions)])
known_values[:, 0] = conditions[:, :DIM]
known_values[:, -1] = conditions[:, DIM:2 * DIM]
known_mask = torch.zeros_like(known_values, dtype=torch.bool)
known_mask[:, (0, -1)] = True
# Именно эта маска использовалась внутри run_navigation при семплировании.
print("Known coordinates per trajectory:", int(known_mask[0].sum()))
print("Generated collision-free fraction:", experiment.metrics["generated_collision_free_fraction"])


## Поле и потенциал

Для выбранной цели аналитическое поле равно `F = −∇U`: квадратичное притяжение плюс гладкое отталкивание от signed distance до стен. Оно может иметь локальные минимумы, поэтому эталонные траектории строятся отдельным планировщиком.

Численное восстановление `recover_potential` выполняется только на доступных вершинах сетки. Выводятся остаток, curl, число компонент и статус решателя. Потенциал задан с независимой аддитивной константой на каждой связной компоненте; малые численные ошибки не доказывают глобальную консервативность поля. Для 3D показаны полноценные траектории/векторы и z-срезы потенциалов.

Для аналитического `−∇U` потенциал известен. Углы стен могут дать большой дискретный curl на грубой сетке: пороги диагностики относятся к семплированным значениям. Уточните разрешение и сравните восстановленное поле с аналитическим U.


In [ ]:
artifacts = render_navigation(experiment, OUTPUT)
for name in (
    "reference_routes", "generated_routes", "navigation_field",
    "analytical_potential", "recovered_potential",
):
    display(Image(filename=str(artifacts[name][0])))
print("PNG, SVG and metrics:", OUTPUT)


## Другая геометрия среды

Стены задаются `[W, 2, D]`: нижний угол, верхний угол. Любой объект `WallNavigation` создаёт тот же `SampleDataset`, поэтому его можно передать в тот же trainer. Разное число стен меняет `condition_dim`; для совместного обучения сцен держите число стен фиксированным или используйте собственный encoder, который принимает набор препятствий.

В 1D сплошное препятствие разрывает коридор: обойти его невозможно. Среда `navigation_1d` по умолчанию учит перемещение в открытом коридоре, а для заданных стен допускает только связанные A/B.


In [ ]:
custom_scene = WallNavigation(
    dim=2, path_length=32,
    walls=[
        [[-0.45, -1.00], [-0.30, 0.45]],
        [[ 0.20, -0.45], [ 0.35, 1.00]],
    ],
)
custom_dataset = custom_scene.sample_dataset(n_samples=6, seed=SEED + 1)
assert custom_scene.collision_free(custom_dataset.samples).all()
print("Custom scene condition dimension:", custom_dataset.condition_dim)
figure, axis = plot_trajectories(
    custom_dataset.samples, walls=custom_scene.walls, title="A* in another wall layout",
)
display(figure)
import matplotlib.pyplot as plt
plt.close(figure)


## Следующие эксперименты

Сравните MLP и Transformer, веса геометрических штрафов, разные схемы conditioning, DDIM/DDPM и rectified flow. Сохраняйте seed, параметры и метрики; оценивайте не только endpoints, но и полную геометрическую допустимость отрезков.

Для GIF все waypoints можно показать как облако; это отображает состояния денойзинга, а не физическое время движения частицы.


In [ ]:
# Опциональный GIF, без ffmpeg:
# from diffusion_lab.visualization import save_denoising_gif
# history = experiment.output.history
# save_denoising_gif(
#     history.reshape(len(history), -1, DIM), OUTPUT / "denoising_waypoints.gif",
#     walls=experiment.scene.walls,
# )
